In [ ]:
from datetime import datetime
from opera_utils import disp
import geopandas as gpd
import os
from pathlib import Path
import subprocess
from transboundary_opera import displacement_tools as dt
import matplotlib.pyplot as plt
import xarray as xr
from shapely.geometry import mapping
import ultraplot as uplt
from pyproj import CRS
import asf_search as asf
import re

# %matplotlib widget
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
SEARCH_START = datetime(2024, 6, 1)
SEARCH_END = datetime(2025, 1, 1)

gdf = gpd.read_file(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")
# frame_ids = dt.get_opera_frame_ids(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")     # this only gets one set of frame ids 

In [ ]:
# Initialize a list to store frame IDs for each row
all_frame_ids = []

for entry, row in gdf.iterrows():
    
    results = asf.geo_search(
        intersectsWith=row.geometry.convex_hull.wkt,
        dataset=asf.DATASET.OPERA_S1,
        processingLevel=asf.PRODUCT_TYPE.DISP_S1,
        maxResults=50  # We only need a few results to identify the Frame IDs
    )
    
    # Set for the current row
    current_frame_ids = set()
    pattern = re.compile(r'_F(\d{5})_')

    for product in results:
        filename = product.properties['fileName']
        match = pattern.search(filename)
        if match:
            frame_id_str = match.group(1) 
            current_frame_ids.add(int(frame_id_str))
    
    # Add the sorted list of frame IDs to our collection
    all_frame_ids.append(sorted(list(current_frame_ids)))

# Assign the collected frame IDs to a new column in the GeoDataFrame
gdf['frame_ids'] = all_frame_ids

In [ ]:
# TODO: attempt refined download for each of the frames that cover each aquifer. May need to clip the bbox of the aquifers and segment by the frame_ids

unique_frame_ids = sorted({fid for ids in gdf['frame_ids'] if isinstance(ids, (list, tuple, set)) for fid in ids})
len(unique_frame_ids)

In [ ]:
disp_products = {}

for frame_id in unique_frame_ids:
    disp_products[frame_id] = disp.search(
        frame_id=frame_id,
        start_datetime=SEARCH_START,
        end_datetime=SEARCH_END,
        url_type="https",
        print_urls=False,
    )

# test_frame_id = unique_frame_ids[21]

# # individual search for one frame id
# disp_products = disp.search(
#         frame_id=test_frame_id,
#         start_datetime=datetime(2024, 1, 1),
#         end_datetime=datetime(2025, 1, 1),
#         url_type="https",
#         print_urls=False,
#     )

# disp_products_stack = disp.DispProductStack(disp_products)

In [ ]:
tot_bytes = 0

for key, list in disp_products.items():
    # print(f"Frame ID: {key}, Number of Products: {len(list)}")
    for disp_product in list:
        tot_bytes += disp_product.size_in_bytes

print(f"\nTotal Storage needed for OPERA DISP Products: {round(tot_bytes / 1073741824, 2)} GB")

In [ ]:
# if you have a .netrc or appropriate envronment variables set up, you can directly open a file or two in memory
# test1 = xr.open_dataset(disp.open_file(url = disp_products[0].filename))
# test2 = xr.open_dataset(disp.open_file(url = disp_products[1].filename))

# Search for data
- Try to figure out how to stream the data instead of just downloading locally? This probably has something to do with the .netrc file

In [ ]:
# out_path = Path('/Users/clayc/Documents/Work/US_Mex_InSAR') / "OPERA_data"
out_path = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data"
os.makedirs(out_path, exist_ok=True)

for key, list in disp_products.items():
    print(f"Frame ID: {key}, Number of Products: {len(list)}")

    download_dir = out_path / str(key)
    os.makedirs(download_dir, exist_ok=True)

    if len(list) == 0:
        print(f"No products found for Frame ID: {key}. Skipping download.")
        continue


    cmd = [
        "opera-utils", "disp-s1-download",
        "--output-dir", str(download_dir),
        "--frame-id", str(key),
        "--start-datetime", SEARCH_START.strftime('%Y-%m-%d'),
        "--end-datetime", SEARCH_END.strftime('%Y-%m-%d'),
        # "--bbox", f"{round(bbox[0],2)}", f"{round(bbox[1],2)}", f"{round(bbox[2],2)}", f"{round(bbox[3],2)}",
        "--url-type", "HTTPS", 
        "--num-workers", "8"
    ]

    print("Starting download...")
    subprocess.run(cmd, check=True)

# Some short time line displacements

In [ ]:
ds_reference = xr.open_dataset(disp.open_file(test_nc1))
ds_secondary = xr.open_dataset(disp.open_file(test_nc2))
difference = ds_secondary.displacement - ds_reference.displacement


unique_frame_ids = sorted({fid for ids in gdf['frame_ids'] if isinstance(ids, (list, tuple, set)) for fid in ids})

In [ ]:
# check which aquifers are covered by frame id 16939
# you can change 16939 to any other frame id to see which aquifers it covers
gdf[gdf['frame_ids'].apply(lambda x: 16939 in x)]

In [ ]:
i = 26
j = 0

# epsg 4326 data
test_aquifer = gdf.iloc[i]
test_frame_ids = test_aquifer['frame_ids']
test_aq_name = "_".join(test_aquifer.AQ_NAME.split(" "))
test_aq_boundary = test_aquifer.geometry
test_aq_bbox = test_aq_boundary.bounds

# utm projected data
test_aquifer_proj = gdf.to_crs(product_stack[j].crs).iloc[i]
test_aq_boundary_proj = test_aquifer_proj.geometry

In [ ]:
# TODO: work on ones that do not overlap

lon_left, lat_bottom, lon_right, lat_top = test_aq_bbox
row_start, col_start = product_stack[j].lonlat_to_rowcol(lon_left, lat_top)
row_stop, col_stop = product_stack[j].lonlat_to_rowcol(lon_right, lat_bottom)
rows = slice(row_start, row_stop)
cols = slice(col_start, col_stop)
print(rows, cols)

In [ ]:
ref_sub = ds_reference.isel(y=rows, x=cols)
sec_sub = ds_secondary.isel(y=rows, x=cols)

difference_sub = sec_sub.displacement - ref_sub.displacement
difference_sub.plot.imshow(cmap="RdBu")

In [ ]:
plt.figure()
unlikely_unwrap_errors = (sec_sub.timeseries_inversion_residuals % 6.28) < 0.01
high_coherence = sec_sub.temporal_coherence > 0.7
difference_sub.where(high_coherence * unlikely_unwrap_errors).plot.imshow(cmap="RdBu");

In [ ]:
plt.figure()
unlikely_unwrap_errors = (sec_sub.timeseries_inversion_residuals % 6.28) < 0.01
high_coherence = sec_sub.temporal_coherence > 0.7
difference_sub.where(high_coherence * unlikely_unwrap_errors).plot.imshow(cmap="RdBu")

# Add the aquifer boundary
ax = plt.gca()
if test_aq_boundary_proj.geom_type == 'Polygon':
    x, y = test_aq_boundary_proj.exterior.xy
    ax.plot(x, y, color='black', linewidth=2, label='Aquifer Boundary')
elif test_aq_boundary_proj.geom_type == 'MultiPolygon':
    for geom in test_aq_boundary_proj.geoms:
        x, y = geom.exterior.xy
        ax.plot(x, y, color='black', linewidth=2)

ax.legend()
plt.tight_layout()

## Interactive Map(s)
- once I figure out how to stream data easily, probably via OSL?, I can then plot some of the results above?

In [ ]:
from ipyleaflet import Map, GeoData, basemaps, LayersControl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Create a color mapping for each unique aquifer name
unique_names = gdf['AQ_NAME'].unique()
cmap = plt.get_cmap('tab20', len(unique_names))
color_dict = {name: mcolors.rgb2hex(cmap(i)) for i, name in enumerate(unique_names)}

# Add color column to gdf
gdf['color'] = gdf['AQ_NAME'].map(color_dict)

# Calculate center from all polygons
center = [gdf.geometry.union_all().centroid.y, gdf.geometry.union_all().centroid.x]

# Create map
m = Map(center=center, zoom=6, basemap=basemaps.OpenStreetMap.Mapnik)

# Add each aquifer with its unique color
for idx, row in gdf.iterrows():
    geo_data = GeoData(
        geo_dataframe=gdf.iloc[[idx]],
        style={'color': row['color'], 'fillColor': row['color'], 
               'fillOpacity': 0.3, 'weight': 2},
        hover_style={'fillOpacity': 0.6},
        name=row['AQ_NAME']
    )
    m.add_layer(geo_data)

m.add_control(LayersControl())

m

# Reformat time series displacements
- download full ts data for each frame within the entire dataset
- subset the downloaded .nc files based on our shapefiles
- bring in the .nc files and restack with disp.DispProductStack.from_file_list
- convert to hdf5 and send to mintpy for better time-series velocities and E-W + U-D decomposition from ascending and descending tracks

In [ ]:
data_dir = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data"

frame_disp_stacks = {}

for frame in os.listdir(data_dir):
    frame_dir = data_dir / str(frame)
    try:
        frame_disp_stacks[frame] = disp.DispProductStack.from_file_list(list(frame_dir.glob('*.nc')))
    except Exception as e:
        print(f"Could not create DispProductStack for frame {frame}: {e}")
        continue

In [ ]:
for key, frame in frame_disp_stacks.items():
    try:
        disp.reformat_stack(
            input_files=frame.filenames,
            output_name=str(data_dir / f"{frame.to_dataframe().frame_id.unique()[0]}_disp_stack.nc"),
            apply_solid_earth_corrections= True,
            apply_ionospheric_corrections= True,
            reference_method = 'high_coherence'    # 'border' or 'high_coherence'
        )
    except Exception as e:
        print(f"Could not reformat stack for frame {key}: {e}")
        continue

# Load in the stacks and visualize time series displacements

In [ ]:
# TODO: try to figure out how to get zonal statistics for the displacement
# TODO: save these stacks in a format that can be used in MintPy (hdf5?)

In [ ]:
data_dir = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data"
disp_stacks = list(data_dir.glob('*_disp_stack.nc'))

In [ ]:
test_stack = disp_stacks[15]
ds = xr.open_dataset(test_stack)

wkt = ds.spatial_ref.crs_wkt
crs = CRS.from_wkt(wkt)

# try the simple route first, fallback to authority tuple
epsg = crs.to_epsg() or (int(crs.to_authority()[1]) if crs.to_authority() else None)

# da_displacement_latlon = ds.displacement.rio.reproject("EPSG:4326")

In [ ]:
i = 18  # index of the above returns
j = 0

# epsg 4326 data
test_aquifer = gdf.iloc[i]
test_frame_ids = test_aquifer['frame_ids']
test_aq_name = "_".join(test_aquifer.AQ_NAME.split(" "))
test_aq_boundary = test_aquifer.geometry
test_aq_bbox = test_aq_boundary.bounds

# utm projected data
test_aquifer_proj = gdf.to_crs(epsg).iloc[i]
test_aq_boundary_proj = test_aquifer_proj.geometry

In [ ]:
test_boundaries = list(gdf[gdf['frame_ids'].apply(lambda x: float(test_stack.stem.replace('_disp_stack', '')) in x)].to_crs(epsg).geometry)

In [ ]:
uplt.rc["image.cmap"] = "RdBu_r"
uplt.rc["image.interpolation"] = "nearest"

da_last = ds.displacement[-1]

strict_mask = (ds.temporal_coherence[-1] > 0.75) | (ds.phase_similarity[-1] > 0.6)

da_last_coarse = (
    # Apply the stricter mask
    da_last.where(strict_mask)
    # Coarsen 3 by 3 to make a 90 meter image
    .coarsen(x=3, y=3, boundary="trim")
    # Take the nan-aware median of each 3x3 window
    .median()
)

fig, ax = uplt.subplots(refwidth=5)
da_last_coarse.plot.imshow(ax=ax, vmax=0.2, vmin=-0.2, cmap="RdBu_r")
ax.set_aspect("equal")

# Add the aquifer boundary
# The key fix: use the coordinates from your DataArray, not the default axis coordinates
for boundary_proj in test_boundaries:
    if boundary_proj.geom_type == 'Polygon':
        x, y = boundary_proj.exterior.xy
        ax.plot(x, y, color='black', linewidth=2, label='Aquifer Boundary', transform=ax.transData)
    elif boundary_proj.geom_type == 'MultiPolygon':
        for geom in boundary_proj.geoms:
            x, y = geom.exterior.xy
            ax.plot(x, y, color='black', linewidth=2)
            # Add label only once for legend
            # ax.plot([], [], color='black', linewidth=2, label='Aquifer Boundary')

            # ax.legend()
# plt.tight_layout()